# Advanced Chemical Reactor Engineering — Group: Stirred not Shaken
### Reactor Simulation: Glucose → Fructose → HMF → FDCA

This notebook covers two reactor concepts for the catalytic oxidation of glucose to FDCA:

**Part A — CSTR** (Continuous Stirred Tank Reactor): transient ODE model, 12 state variables, reaching steady state.
**Part B — SDR** (Spinning Disc Reactor): N ideal CSTRs in series, approximating plug-flow behaviour with high mass transfer.

Both reactors share the same reaction network, kinetic parameters, fluid properties, and operating conditions.
Reactor-specific sections (hydrodynamics, mass transfer correlations, ODE formulation) are kept separate to reflect the different physical reality.

### Reaction pathway
$$\text{Glucose} \xrightarrow{k_1} \text{Fructose} \xrightarrow{k_3} \text{HMF} \xrightarrow{k_6} \text{FDCA}$$

Side reactions from HMF:
- $\text{HMF} \xrightarrow{k_4} \text{Humins}$
- $\text{HMF} \xrightarrow{k_5} \text{Levulinic acid + Formic acid}$

### Three-phase system
The reactor operates as a gas–liquid–solid (G-L-S) system:
- Gas phase: $O_2$ bubbles
- Liquid phase: aqueous (sugars/HMF) + organic MIBK solvent (HMF extraction)
- Solid phase: catalyst particles


### 0. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


### 1. Physical Properties

All properties are evaluated at the operating temperature T = 140 °C (413.15 K).

#### Diffusivities — Wilke–Chang correlation
The diffusivity is corrected from a reference value at 298 K to the operating conditions using:
$$D(T, \mu) = D_{\text{ref}} \cdot \frac{T}{T_{\text{ref}}} \cdot \frac{\mu_{\text{ref}}}{\mu}$$

This accounts for both the increase in thermal energy (higher T → faster diffusion) and the decrease in viscosity at elevated temperature (lower μ -> less resistance to molecular motion).

In [ ]:
T_C = 160.0                         # degree C
T   = T_C + 273.15                  # K
g   = 9.81                          # m/s^2
R   = 8.3145                        # J/mol/K
 
rho = 819.2 - 0.917 * T_C                   # kg/m^3    density
A_andrade = 1.490e-5                        # Pa·s      coefficient A
B_andrade = 1159.4                          # K         coefficient B
mu = A_andrade * np.exp(B_andrade / T)      # Pa·s      dynamic viscosity  
nu = mu / rho                               # m^2/s     kinematic viscosity
 
sigma = 12e-3                               # N/m  MIBK/water
 
# Diffusivities corrected to operating T and mu via Wilke-Chang
phi = 1.0                               # -         association factor
M_B = 100.16                            # g/mol     molecular weight MIBK
 
V_HMF = 6*14.8 + 6*3.7 + 3*7.4 - 11.5           # cm^3/mol  molar volume HMF
V_O2 = 2*7.4 + 7                                # cm^3/mol  molar volume O2
V_FDCA = 6*14.8 + 4*3.7 + 4*12 + 7.4 - 11.5     # cm^3/mol  molar volume FDCA
 
mu_mPas = mu * 1e3                      # convert Pa.s -> mPa.s for Wilke-Chang
 
D_HMF = 7.4e-12 * (phi * M_B)**0.5 * T / (mu_mPas * V_HMF**0.6)  # m_org^2/s
D_O2  = 7.4e-12 * (phi * M_B)**0.5 * T / (mu_mPas * V_O2**0.6)   # m_org^2/s
D_FDCA  = 7.4e-12 * (phi * M_B)**0.5 * T / (mu_mPas * V_FDCA**0.6)   # m_org^2/s
 
print(f"T          = {T:.2f} K  ({T_C:.1f} C)")
print(f"rho        = {rho:.1f} kg/m^3")
print(f"mu         = {mu*1e3:.4f} mPa.s")
print(f"nu         = {nu:.3e} m^2/s")
print(f"V_HMF      = {V_HMF:.1f} cm^3/mol")
print(f"V_O2       = {V_O2:.1f} cm^3/mol")
print(f"D_HMF      = {D_HMF:.3e} m^2/s")
print(f"D_O2       = {D_O2:.3e} m^2/s")
print(f"D_FDCA = {D_FDCA:.3e} m_org^2/s")

### 2. Reactor Geometry and Phase Volume Fractions

The 100 $m^3$ reactor is divided into four phases. Volume fractions are defined as $m^3_{\text{phase}} / m^3_{\text{reactor}}$:

| Phase | Symbol | Fraction | Volume (m³) | Units (Fractional) |
|-------|--------|----------|-------------|--------------------|
| Gas (O₂ bubbles) | $\varepsilon_{\text{g}}$ | 0.20 | 20 | $m_{\text{g}}^3 / m_{\text{R}}^3$ |
| Catalyst particles | $\varepsilon_{\text{s}}$ | 0.10 | 10 | $m_{\text{s}}^3 / m_{\text{R}}^3$ |
| Ral liquid (aq + org) | $\varepsilon_{\text{l}}$ | 0.70 | 70 | $m_{\text{L}}^3 / m_{\text{R}}^3$ |
| &nbsp;&nbsp;Aqueous (1:2 split) | $\varepsilon_{\text{aq}}$ | 0.233 | 23.33 | $m_{\text{aq}}^3 / m_{\text{L}}^3$ |
| &nbsp;&nbsp;Organic MIBK (1:2 split) | $\varepsilon_{\text{org}}$ | 0.467 | 46.67 | $m_{\text{org}}^3 / m_{\text{L}}^3$ |

The **aqueous:organic ratio of 1:2** reflects the extraction design — HMF partitions preferentially into MIBK, which drives the dehydration equilibrium forward.

The agitator diameter follows the standard rule $D_A / D_T = 1/3$ for Rushton turbines.

In [ ]:
V_total = 100.0                     # m_R^3 reactive volume
eps_g = 0.20                        # m_g^3/m_R^3 bubble volume fraction
eps_s = 0.10                        # m_s^3/m_R^3 particles volume fraction
eps_l_set = 1.0 - eps_g - eps_s     # m_l^3/m_R^3 total liquids volume fraction
eps_aq = 1/3                        # m_aq^3/m_l^3 aquatic phase faction
eps_org = 1 - eps_aq                # m_org^3/m_l^3 organic phase fraction

V_g   = eps_g * V_total         # m_g^3   bubble gas phase
V_p   = eps_s * V_total         # m_s^3   catalyst particles
V_liq = eps_l_set * V_total     # m_l^3   total liquid (aq + org)
V_aq  = V_liq * eps_aq          # m_aq^3   aqueous phase
V_org = V_liq * eps_org         # m_org^3   organic phase

D_T  = (4 * V_total / np.pi)**(1/3)    # m    tank diameter
D_A  = D_T / 3                         # m    agitator (D_A/D_T = 1/3)
A_cs = np.pi / 4 * D_T**2              # m^2  cross-section

C_Glu_feed = 1500.0   # mol/m_aq^3  glucose feed concentration
m_AO = 0.77           # [-]  HMF partition coefficient aq -> org <====== check the units of this partition coefficient!!!

tau   = 3600          # s residence time (1 h)
F_aq  = V_aq  / tau   # m_aq^3/s  aqueous flow rate
F_org = V_org / tau   # m_org^3/s organic flow rate
F_g   = V_g / tau     # m_g^3/s   gas flow rate  

print(f"Residence time tau = {tau/3600:.1f} h")
print(f"F_aq  = {F_aq*1000:.2f} L/s")
print(f"F_org = {F_org*1000:.2f} L/s")
print(f"F_org = {F_g*1000:.2f} L/s\n")

print(f"Tank diameter:   D_T = {D_T:.2f} m")
print(f"Agitator diam:   D_A = {D_A:.2f} m")
print(f"Phase volumes:   V_g={V_g:.1f}  V_aq={V_aq:.2f}  V_org={V_org:.2f}  V_p={V_p:.1f} m^3")
print(f"Sum check:       {V_g+V_aq+V_org+V_p:.2f} m^3  (should be {V_total})")
d_p   = 5e-6             # m_s   catalyst particle diameter


### 3. Reaction Kinetics

#### Isomerisation and dehydration (aqueous phase)
All rate constants are assumned to be first-order ($s^{-1}$):

| Reaction | Symbol | Value (s⁻¹) |
|----------|--------|-------------|
| Glucose $\rightarrow$ Fructose | $k_1$ | 0.104/60 |
| Fructose $\rightarrow$ Glucose (reverse) | $k_2$ | 0.052/60 |
| Fructose $\rightarrow$ HMF | $k_3$ | 0.286/60 |
| HMF $\rightarrow$ Humins | $k_4$ | 0.013/60 |
| HMF $\rightarrow$ LA + FA | $k_5$ | 0.031/60 |

#### Oxidation to FDCA (catalyst particle surface)
HMF oxidation proceeds via two parallel series paths:
- Path A: HMF $\rightarrow$ DFF $\rightarrow$ FFCA $\rightarrow$ FDCA
- Path B: HMF $\rightarrow$ HFCA $\rightarrow$ FFCA $\rightarrow$ FDCA

For each path, the series steps are combined using the resistance-in-series formula $1/k_{\text{eff}} = 1/k_1 + 1/k_2$, then the two parallel paths are added:
$$k_6 = \frac{1}{\frac{1}{k_{\text{HMF→DFF}}} + \frac{1}{k_{\text{DFF→FFCA}}}} + \frac{1}{\frac{1}{k_{\text{HMF→HFCA}}} + \frac{1}{k_{\text{HFCA→FFCA}}}}$$

#### Thiele modulus and internal effectiveness factor
For a spherical catalyst particle with 1st-order reaction in HMF, the Thiele modulus is:
$$\phi = \frac{d_p}{6}\sqrt{\frac{k_6}{D_{\text{HMF}}}}$$

The internal effectiveness factor (ratio of actual to maximum reaction rate) is:
$$\eta = \frac{1}{\phi}\left(\frac{1}{\tanh(3\phi)} - \frac{1}{3\phi}\right)$$

When $\phi \ll 1$: $\eta \approx 1$ → no internal diffusion limitation.

In [ ]:
k1, k2 = 0.104/60, 0.052/60    # s^-1  Glu <-> Fru (isomerisation)
k3      = 0.286/60             # s^-1  Fru -> HMF  (dehydration)
k4      = 0.013/60             # s^-1  HMF -> Humins
k5      = 0.031/60             # s^-1  HMF -> LA + FA

# Oxidation: two parallel series paths (resistance-in-series + parallel combination)
# Path A (via DFF):   1/k_eff_A = 1/k_HMF_DFF + 1/k_DFF_FFCA
# Path B (via HFCA):  1/k_eff_B = 1/k_HMF_HFCA + 1/k_HFCA_FFCA
k6 = 1/(1/0.0556 + 1/0.01576) + 1/(1/0.0298 + 1/1.58e-3)   # s^-1 Overall reaction rate on catalyst
k6_ = k6 

# Thiele modulus & internal effectiveness factor (sphere, 1st order)
phi = (d_p / 6) * np.sqrt(k6 / D_HMF)
eta = (1/np.tanh(3*phi) - 1/(3*phi)) / phi

print(f"k6  (oxidation, combined) = {k6:.4f} s^-1")
print(f"Thiele modulus  phi = {phi:.5f}")
print(f"Effectiveness   eta = {eta:.6f}  (phi << 1 -> no internal diffusion limitation)")

### 4. O₂ Saturation Concentration — Henry's Law

The driving force for gas–liquid O₂ transfer is $C_{O_2}^* - C_{O_2,\text{org}}$, where $C_{O_2}^*$ is the saturation concentration set by Henry's law.

#### Vapour pressures (Antoine equation)
The total pressure is 10 bar; the partial pressure of O₂ is:
$$P_{O_2} = P_{\text{total}} - P_{\text{vap,water}} - P_{\text{vap,MIBK}}$$

#### Henry's constant temperature correction (van't Hoff)
Henry's constant increases with temperature (O₂ becomes less soluble at higher T):
$$H(T) = H_{298} \cdot \exp\!\left[\frac{\Delta H_{\text{sol}}}{R}\left(\frac{1}{298} - \frac{1}{T}\right)\right]$$

#### Saturation concentration
$$C_{O_2}^* = \frac{P_{O_2}}{H(T)} \quad \left[\frac{mol}{m_\text{org}^3}\right]$$

In [ ]:
# Vapour pressures via Antoine equation
P_O2      = 10e5                                            # Pa    chosen partial pressure O2
P_aq_vap  = 10**(3.55959 - 643.748/(T - 198.043))*1e5       # Pa    partial pressure aqueous phase            
P_org_vap = 10**(3.95298 - 1254.095/(T - 71.537))*1e5       # Pa    partial pressure organic phase      
P_O2_min  = P_aq_vap + P_org_vap                            # Pa    minimum partial pressure of O2 gas required for flux into liquid phases
P_tot_min = P_aq_vap + P_org_vap + P_O2_min                 # Pa    total pressure in reactor required for it to work
  
# Henry's constant at 298 K, corrected to operating T
H_O2_298  = 101.3e3 / (8.71e-4 * (780e3/58.08))             # Pa.m_org^3/mol    solubility at 25C
H_O2      = H_O2_298 * np.exp(15e3/R * (1/298 - 1/T))       # Pa.m_org^3/mol    solubility at 160C
C_O2_sat  = P_O2 / H_O2                                     # mol/m_org^3       saturation concentration
 
print(f"P_vap water   = {P_aq_vap:>10.2e} Pa")
print(f"P_vap MIBK    = {P_org_vap:>10.2e} Pa\n")
print(f"P_O2 Minimal  = {P_O2_min:>10.2e} Pa")
print(f"P_tot Minimal = {P_tot_min:>10.2e} Pa\n")
print(f"P_O2 Chosen   = {P_O2:>10.2e} Pa\n")
print(f"H_O2          = {H_O2:>10.2e} Pa.m_org^3/mol")
print(f"C*_O2         = {C_O2_sat:>10.2e} mol/m_org^3")

if P_O2 < P_O2_min:
    raise ValueError(f"Chosen O2 partial pressure lower than vapour pressure. System will have no O2 flux to reaction")

---
## Part A — CSTR (Continuous Stirred Tank Reactor)

The CSTR model solves a transient ODE system (12 state variables) from startup to steady state.
Mass transfer coefficients are calculated from first-principles correlations:
- Gas–liquid (O₂ absorption): Yagi & Yoshida (1975)
- Liquid–liquid and liquid–solid: Armenante & Kirwan (1989)


### 5. CSTR — Agitation, Gas Flow, and Power Input

#### Minimum agitation speed
The impeller must rotate fast enough to disperse gas bubbles. The minimum speed $N_{RM}$ (Calderbank criterion) scales with tank/agitator geometry:
$$N_{RM} = 0.22 \cdot \frac{D_T^{1.5}}{D_A^2}$$

#### Power dissipation
Total power input has two contributions:

1. Impeller power (6-blade Rushton turbine, power number $N_p = 5$):
$$P_L = N_p \cdot \rho \cdot N_R^3 \cdot D_A^5$$

2. Gas expansion power (isothermal expansion of rising bubbles):
$$P_G = v_{SG} \cdot \rho \cdot g \cdot V_{\text{total}}$$

The specific energy dissipation rate $\varepsilon = P/V\rho$ ($m^2$/$s^3$) governs turbulent mass transfer and is used in the mass transfer correlations below.

In [ ]:
N_p =  5.0                          # -      wanted power number (6-blade Rushton turbine)

v_SG = 4 * F_g / (np.pi*D_T**2)     # m/s     superficial gas velocity
P_G  = v_SG * rho * g * V_total     # W       gas power
PV = (eps_g / 0.27 / v_SG**0.67) ** (1/0.31) # power per unit of volume
P_L  = PV*V_liq -  P_G              # W           impeller power

N_RM = 0.22 * D_T**1.5 / D_A**2           # rev/s  minimal needed speed (Calderbank)
N_R  = (P_L / rho / N_p / D_A**5)**(1/3)  # rev/s  minimal wanted stirrer angular speed
print(f"N_R = {N_R:.2f} rev/s  >  N_RM = {N_RM:.2f} rev/s: {N_R > N_RM}")

eps  = PV / rho                       # m_l^2/s^3     specific energy dissipation

# Gas holdup and bubble diameter (Calderbank correlations)
d_b   = 4.15*(sigma/rho)**0.6 * (P_L/V_total)**(-0.4) * v_SG**0.5 + 9e-4  # m_g  surface average bubble diameter (Sauter Diameter)

print(f"P/V  = {PV:.1f} W/m^3")
print(f"eps  = {eps:.4f} m^2/s^3")
print(f"eps_g (gas holdup) = {eps_g:.3f}")
print(f"d_b  (bubble diam) = {d_b*1e3:.2f} mm_g")

### 6. CSTR — Mass Transfer Coefficients

Three interfacial mass transfer steps are modelled, each with a volumetric coefficient $k_La$ ($s^{-1}$).

#### A1 — Gas → Organic liquid (O₂ absorption)
Yagi & Yoshida (1975) correlation for the liquid-film coefficient:
$$k_La_{GL} = 0.06 \cdot Re^{1.5} \cdot Fr^{0.19} \cdot Sc^{0.5} \cdot Ca^{0.6} \cdot V_r^{0.32} \cdot \frac{D_{O_2}}{D_A^2}$$

Dimensionless groups:

$Re = N_R D_A^2 \rho/\mu$

$Fr = N_R^2 D_A/g$

$Sc = \mu/(\rho D)$

$Ca = \mu v_{SG}/\sigma$ 

$V_r = N_R D_A/v_{SG}$

The interfacial area per unit volume is:
$$a_{GL} = \frac{6\,\varepsilon_g}{d_b}$$

#### A2 — Aqueous → Organic liquid (HMF and FDCA)
Armenante & Kirwan (1989) — Sherwood number for droplets in turbulent flow:
$$Sh = 2 + 0.36 \cdot Re_{LL}^{0.75} \cdot Sc^{0.33}$$
where $Re_{LL} = (\varepsilon\, d_{\text{drop}}^4/\nu^3)^{0.25}$ is a turbulence-based Reynolds number.

Droplet diameter from the turbulent Kolmogorov length scale:
$$d_{\text{drop}} = \left(\frac{\sigma}{\rho\,\varepsilon^{2/3}}\right)^{0.6}$$

#### A3 — Organic liquid → Catalyst particle (HMF, $O_2$ and FDCA)
Same Armenante & Kirwan correlation applied to solid particles of diameter $d_p = 5\,\mu\text{m}_\text{s}$:
$$a_{LS} = \frac{6\,\varepsilon_s}{d_p}$$

In [ ]:
# Helper: Armenante & Kirwan (1989) Sherwood number
def Sh_AK(eps, d, nu, mu, rho, D):
    Re = (eps * d**4 / nu**3)**0.25
    Sh = 2 + 0.36 * Re**0.75 * (mu / (rho * D))**0.33
    return Sh

# A1: Gas-liquid (O2: gas -> organic) — Yagi & Yoshida (1975)
kLa_GL = (0.06
          * (N_R * D_A**2 * rho / mu)**1.5       # Re
          * (N_R**2 * D_A / g)**0.19              # Fr
          * (mu / (rho * D_O2))**0.5              # Sc
          * (mu * v_SG / sigma)**0.6              # Ca
          * (N_R * D_A / v_SG)**0.32              # Vr
          * D_O2 / D_A**2)
a_GL = 6 * eps_g / d_b   # m_int^2/m_R^3

# A2: Liquid-liquid (HMF & FDCA: aq -> organic) — Armenante & Kirwan (1989)
d_drop = np.clip((sigma / (rho * eps**(2/3)))**0.6, 2e-4, 3e-3)  # m_aq
a_LL   = 6 * eps_aq / d_drop                                     # m_aq^2/m_org^3

kLa_LL_HMF  = Sh_AK(eps, d_drop, nu, mu, rho, D_HMF)  * D_HMF  / d_drop * a_LL  # s^-1
kLa_LL_FDCA = Sh_AK(eps, d_drop, nu, mu, rho, D_FDCA) * D_FDCA / d_drop * a_LL  # s^-1

# A3: Liquid-solid (HMF & O2: organic -> catalyst) — Armenante & Kirwan (1989)
d_p  = 5e-6                     # m_p  catalyst diameter
a_LS = 6 * eps_s / d_p          # m_int^2/m_cat^3

kLa_LS_HMF = Sh_AK(eps, d_p, nu, mu, rho, D_HMF) * D_HMF / d_p * a_LS  # s^-1
kLa_LS_O2  = Sh_AK(eps, d_p, nu, mu, rho, D_O2)  * D_O2  / d_p * a_LS  # s^-1
kLa_LS_FDCA  = Sh_AK(eps, d_p, nu, mu, rho, D_FDCA)  * D_FDCA  / d_p * a_LS  # s^-1

print(f"kLa_GL      (O2   gas->org) = {kLa_GL:.4f} s^-1")
print(f"kLa_LL_HMF  (HMF  aq->org)  = {kLa_LL_HMF:.4f} s^-1")
print(f"kLa_LL_FDCA (FDCA aq->org)  = {kLa_LL_FDCA:.4f} s^-1")
print(f"kLa_LS_HMF  (HMF  org->cat) = {kLa_LS_HMF:.2f}  s^-1")
print(f"kLa_LS_O2   (O2   org->cat) = {kLa_LS_O2:.2f}  s^-1")
print(f"kLa_LS_FDCA (FDCA org->cat) = {kLa_LS_FDCA:.2f}  s^-1")

### 7. CSTR — ODE System — 12 State Variables

The reactor contains four phases: aqueous, organic (MIBK), catalyst particle, and gas.
Each species in each phase is treated as a separate component — $C_{\text{HMF,aq}} \neq C_{\text{HMF,org}}$.

The general steady-state mole balance for any phase is:

$$\frac{dC_i}{dt} = \underbrace{\frac{F}{V}(C_{i,\text{feed}} - C_i)}_{\text{convective flow}} + \underbrace{\sum r_j}_{\text{reaction}} \pm \underbrace{J_{\text{MT}}}_{\text{mass transfer}}$$

#### State variables
| #  | Symbol | Phase | Species |
|----|--------|-------|---------|
| 0  | $C_{\text{Glu}}$      | aqueous  | Glucose               |
| 1  | $C_{\text{Fru}}$      | aqueous  | Fructose              |
| 2  | $C_{\text{HMF,aq}}$   | aqueous  | HMF                   |
| 3  | $C_{\text{HMF,org}}$  | organic  | HMF                   |
| 4  | $C_{\text{HMF,p}}$    | catalyst | HMF                   |
| 5  | $C_{O_2\text{,org}}$  | organic  | O₂                    |
| 6  | $C_{O_2\text{,p}}$    | catalyst | O₂                    |
| 7  | $C_{\text{FDCA,p}}$   | catalyst | FDCA                  |
| 8  | $C_{\text{FDCA,org}}$ | organic  | FDCA                  |
| 9  | $C_{\text{FDCA,aq}}$  | aqueous  | FDCA                  |
| 10 | $C_{\text{Hum}}$      | aqueous  | Humins                |
| 11 | $C_{\text{LA}}$       | aqueous  | Levulinic acid (= FA) |

---

#### Mole balances

**Aqueous phase** — well-mixed CSTR ($\tau = \tau_{\text{aq}} = V_{\text{aq}}/F_{\text{aq}} = \tau_{\text{org}} = V_{\text{org}}/F_{\text{org}}$):

$$\frac{dC_{\text{Glu}}}{dt} = \frac{F_{\text{aq}}}{V_{\text{aq}}}(C_{\text{Glu,feed}} - C_{\text{Glu}}) - k_1 C_{\text{Glu}} + k_2 C_{\text{Fru}}$$

$$\frac{dC_{\text{Fru}}}{dt} = \frac{F_{\text{aq}}}{V_{\text{aq}}}(0 - C_{\text{Fru}}) + k_1 C_{\text{Glu}} - k_2 C_{\text{Fru}} - k_3 C_{\text{Fru}}$$

$$\frac{dC_{\text{HMF,aq}}}{dt} = \frac{F_{\text{aq}}}{V_{\text{aq}}}(0 - C_{\text{HMF,aq}}) + k_3 C_{\text{Fru}} - k_4 C_{\text{HMF,aq}} - k_5 C_{\text{HMF,aq}} - J_{LL}$$

$$\frac{dC_{\text{Hum}}}{dt} = \frac{F_{\text{aq}}}{V_{\text{aq}}}(0 - C_{\text{Hum}}) + k_4 C_{\text{HMF,aq}}$$

$$\frac{dC_{\text{LA}}}{dt} = \frac{F_{\text{aq}}}{V_{\text{aq}}}(0 - C_{\text{LA}}) + k_5 C_{\text{HMF,aq}}$$

**Organic phase** (MIBK) — $J_{LL}$ rescaled from aqueous basis; $J_{\text{HMF}}$ and $J_{O_2}$ on organic basis:

$$\frac{dC_{\text{HMF,org}}}{dt} = \frac{F_{\text{org}}}{V_{\text{org}}}(0 - C_{\text{HMF,org}}) + J_{LL}\frac{V_{\text{aq}}}{V_{\text{org}}} - J_{\text{HMF}}$$

$$\frac{dC_{O_2\text{,org}}}{dt} = \frac{F_{\text{org}}}{V_{\text{org}}}(0 - C_{O_2\text{,org}}) + J_{GL} - J_{O_2}$$

**Catalyst particle** (no convective flow) — $J_{\text{HMF}}$ and $J_{O_2}$ rescaled from organic basis:

$$\frac{dC_{\text{HMF,p}}}{dt} = J_{\text{HMF}}\frac{V_{\text{org}}}{V_p} - k_6\,\eta\,C_{\text{HMF,p}}\,C_{O_2\text{,p}}$$

$$\frac{dC_{O_2\text{,p}}}{dt} = J_{O_2}\frac{V_{\text{org}}}{V_p} - k_6\,\eta\,C_{\text{HMF,p}}\,C_{O_2\text{,p}}$$

$$\frac{dC_{\text{FDCA,p}}}{dt} = k_6\,\eta\,C_{\text{HMF,p}}\,C_{O_2\text{,p}} + J_\text{FDCA,PO}\frac{V_{\text{org}}}{V_p}$$
$$\frac{dC_{\text{FDCA,org}}}{dt} = - \frac{F_{\text{org}}}{V_{\text{org}}}C_{\text{FDCA}} - J_\text{FDCA,OP} + J_\text{FDCA,LL}\frac{V_\text{aq}}{V_\text{org}}$$
$$\frac{dC_{\text{FDCA,aq}}}{dt} = - \frac{F_{\text{aq}}}{V_{\text{aq}}}C_{\text{FDCA}} - J_\text{FDCA,LL}$$

---

#### Mass transfer fluxes $[\text{mol}\,\text{m}^{-3}_{\text{source}}\,\text{s}^{-1}]$

The driving force for each flux is the concentration difference between bulk phases, corrected
by the partition coefficient $m$ where phases are thermodynamically distinct. Setting
$m^{OP} = 1$ for both HMF and O₂ at the organic–particle interface reflects the assumption
that the catalyst pores are wetted by the same organic solvent — no thermodynamic jump occurs there.

$$J_{GL}  = k_La_{GL}\!\left(C_{O_2}^* - C_{O_2\text{,org}}\right) \qquad\qquad\qquad\qquad \text{O}_2\text{: gas} \to \text{organic}$$

$$J_{LL}  = k_La_{\text{LL,HMF}}\!\left(C_{\text{HMF,aq}} - m_{AO}\,C_{\text{HMF,org}}\right) \qquad \text{HMF: aq} \to \text{organic},\quad m_{AO} = 0.77$$

$$J_{\text{HMF}} = k_La_{\text{LS,HMF}}\!\left(C_{\text{HMF,org}} - C_{\text{HMF,p}}\right) \qquad\qquad \text{HMF: organic} \to \text{catalyst},\quad m^{OP}_{HMF} = 1$$

$$J_{O_2} = k_La_{\text{LS,O}_2}\!\left(C_{O_2\text{,org}} - C_{O_2\text{,p}}\right) \qquad\qquad\qquad\qquad \text{O}_2\text{: organic} \to \text{catalyst},\quad m^{OP}_{O_2} = 1$$

$$J_{\text{FDCA,LL}} = k_La_{\text{LL,FDCA}}\!\left(C_{\text{FDCA,aq}} - C_{\text{FDCA,org}}\right) \qquad \text{FDCA}\text{: aquatic} \to \text{organic}, \quad m^{LL}_{\text{FDCA}} = 1$$

$$J_{\text{FDCA,OP}} = k_La_{\text{LS,FDCA}}\!\left(C_{\text{FDCA,org}} - C_{\text{FDCA,p}}\right) \qquad \text{FDCA}\text{: organic} \to \text{catalyst}, \quad m^{OP}_{\text{FDCA}} = 1$$

> **Volume-ratio prefactors** — fluxes $J$ are computed per m³ of the *source* phase. When a flux
> appears in the balance of a *different* phase it is multiplied by $V_{\text{source}}/V_{\text{destination}}$
> to conserve moles. For example, $J_{LL}$ ($\text{mol}\,\text{m}^{-3}\,\text{ s}^{-1}$) is multiplied by $V_{\text{aq}}/V_{\text{org}}$
> before entering the organic-phase balance; likewise $J_{\text{HMF}}$ and $J_{O_2}$ are multiplied by
> $V_{\text{org}}/V_p$ before entering the particle balance.

In [ ]:
def odes(t, y):
    Glu, Fru, HMF_aq, HMF_org, HMF_p, O2_org, O2_p, FDCA_p, FDCA_org, FDCA_aq, Hum, LA = \
        [max(v, 0.0) for v in y]

    # Reaction rates [mol/m^3_phase/s]
    r1  = k1 * Glu                  # Glucose -> Fructose
    r2  = k2 * Fru                  # Fructose -> Glucose (reverse)
    r3  = k3 * Fru                  # Fructose -> HMF
    r4  = k4 * HMF_aq               # HMF -> Humins
    r5  = k5 * HMF_aq               # HMF -> LA + FA
    r6  = k6 * eta * HMF_p * O2_p   # HMF oxidation on catalyst (eta = effectiveness factor)

    # Mass transfer fluxes [mol/m^3_source/s]
    J_GL      = kLa_GL * (C_O2_sat - O2_org)            # O2:   gas  -> organic
    J_LL      = kLa_LL_HMF * (HMF_aq - m_AO * HMF_org)      # HMF:  aq   -> organic
    J_HMF     = kLa_LS_HMF * (HMF_org - HMF_p)          # HMF:  org  -> catalyst
    J_O2      = kLa_LS_O2 * (O2_org - O2_p)             # O2:   org  -> catalyst
    J_FDCA_LL = kLa_LL_FDCA * (FDCA_org - FDCA_p)       # FDCA: aq   -> org
    J_FDCA_LS = kLa_LS_FDCA * (FDCA_org - FDCA_p)       # FDCA: org  -> catalyst

    # Mole balances: dC/dt = (F/V)*(C_in - C) + reaction +/- mass transfer
    dGlu      = (F_aq/V_aq) * (C_Glu_feed - Glu) + (-r1 + r2)
    dFru      = (F_aq/V_aq) * (0 - Fru) + (r1 - r2 - r3)
    dHMFaq    = (F_aq/V_aq) * (0 - HMF_aq) + (r3 - r4 - r5) - J_LL
    dHMForg   = (F_org/V_org) * (0 - HMF_org) + J_LL*(V_aq/V_org) - J_HMF
    dO2org    = (F_org/V_org) * (0 - O2_org) + J_GL - J_O2
    dHMFp     = J_HMF * (V_org/V_p) - r6
    dO2p      = J_O2 * (V_org/V_p) - r6
    dFDCA_p   = r6 + J_FDCA_LS*(V_org/V_p)
    dFDCA_org = (F_org/V_org) * (0 - FDCA_org) - J_FDCA_LS + J_FDCA_LL*(V_aq/V_org)
    dFDCA_aq  = (F_aq/V_aq) * (0 - FDCA_aq) - J_FDCA_LL
    dHum      = (F_aq/V_aq) * (0 - Hum) + r4
    dLA       = (F_aq/V_aq) * (0 - LA) + r5

    return [dGlu, dFru, dHMFaq, dHMForg, dHMFp, dO2org, dO2p, dFDCA_p, dFDCA_org, dFDCA_aq, dHum, dLA]

### 8. CSTR — Rate-Determining Step (RDS) Analysis

To identify the bottleneck in the process chain, we compute the maximum possible rate for each step assuming all upstream steps are infinitely fast (reactants accumulate to their maximum value).

The step with the smallest maximum rate is the rate-determining step.

| Step | Assumption | Max driving force |
|------|-----------|------------------|
| $k_1$ Glu→Fru | Nothing consumed | $C_{\text{Glu,feed}}$ |
| $k_3$ Fru→HMF | Iso. equilibrium | $C_{\text{Fru,max}} = \frac{k_1/k_2}{1+k_1/k_2}\,C_{\text{Glu,feed}}$ |
| LL HMF aq→org | All Glu→HMF, blocked in aq | $C_{\text{HMF,aq}} = C_{\text{Glu,feed}}$ |
| LS HMF org→cat | Partition eq. from aq | $C_{\text{HMF,org}} = m_{AO}\,C_{\text{Glu,feed}}$ |
| GL O₂ gas→org | Max Henry driving force | $C_{O_2}^*$ |
| LS O₂ org→cat | GL is fast | $C_{O_2}^*$ |
| $k_6$ reaction | Both reactants at max | $C_{\text{HMF,p}} = C_{\text{HMF,org,max}}$, $C_{O_2,p} = C_{O_2}^*$ |

In [ ]:
K_eq = k1 / k2                                      # isomerisation equilibrium constant
C_Fru_max = K_eq / (1 + K_eq) * C_Glu_feed          # max fructose (mass balance)
C_HMF_aq_max = C_Glu_feed                           # all carbon as HMF
C_HMF_org_max = m_AO * C_HMF_aq_max                 # partition equilibrium

rds = {
    'k1  Glu->Fru  (aq)':  k1 * C_Glu_feed * V_aq,
    'k3  Fru->HMF  (aq)':  k3 * C_Fru_max * V_aq,
    'LL  HMF aq->org':     kLa_LL_HMF * C_HMF_aq_max * V_aq,
    'LS  HMF org->cat':    kLa_LS_HMF * C_HMF_org_max * V_org,
    'GL  O2  gas->org':    kLa_GL * C_O2_sat * V_org,
    'LS  O2  org->cat':    kLa_LS_O2 * C_O2_sat * V_org,
    'k6*eta  reaction':    k6*eta * C_HMF_org_max * C_O2_sat * V_p,
}

print("="*55)
print("RDS -- MAXIMUM DRIVING FORCE [mol/s]")
print("="*55)
min_rate = min(rds.values())
for name, rate in rds.items():
    tag = "  <-- BOTTLENECK" if rate == min_rate else ""
    print(f"  {name:25s}  {rate:.3e}{tag}")

### 9. CSTR — Integration — Reaching Steady State

The BDF (Backward Differentiation Formula) solver is used. The simulation runs for 10 residence times to ensure steady state is reached.

Initial conditions: reactor filled with fresh glucose feed; O₂ at saturation throughout liquid.

In [ ]:
sim_time = 10
view_time = 1
y0  = [C_Glu_feed, 0, 0, 0, 0, C_O2_sat, C_O2_sat, 0, 0, 0, 0, 0]
sol = solve_ivp(odes, [0, sim_time*tau], y0, method='BDF', rtol=1e-9, atol=1e-11, dense_output=True)

t_plot = np.linspace(0, view_time*tau, 2000)
y_plot = sol.sol(t_plot) 
ss = sol.y[:, -1]

Glu_ss, Fru_ss, HMFaq_ss, HMForg_ss, HMFp_ss, O2org_ss, O2p_ss, FDCAp_ss, FDCAorg_ss, FDCAaq_ss, Hum_ss, LA_ss = ss

print(f"Integration successful: {sol.success}")
print(f"Steady-state concentrations [mol/m_phase^3]:")
print(f"  Glu      = {Glu_ss:.1f}")
print(f"  Fru      = {Fru_ss:.1f}")
print(f"  HMF_aq   = {HMFaq_ss:.3f}")
print(f"  HMF_org  = {HMForg_ss:.3f}")
print(f"  O2_org   = {O2org_ss:.4f}")
print(f"  FDCA_org = {FDCAorg_ss:.3f}")
print(f"  FDCA_aq  = {FDCAaq_ss:.3f}")
print(f"  Humins   = {Hum_ss:.3f}")
print(f"  LA=FA    = {LA_ss:.3f}")

### 10. CSTR — Steady-State Performance Metrics

#### Glucose conversion
$$X_{\text{Glu}} = \frac{C_{\text{Glu,feed}} - C_{\text{Glu,SS}}}{C_{\text{Glu,feed}}}$$

#### Selectivity (molar basis)
$$S_i = \frac{R_{\text{d}}}{\Sigma_i R_{\text{u,i}}}$$

Two multiple selectivities can be calculated. The most important ones are the selectivity of $R_1, R_6$, where for $R_1$ the selectivity is only calculated over the whole reaction step, while for $R_6$ the selectivity is calculated against the whole reaction networks or the "parallel reactions" $R_4$ and $R_5$.

In [ ]:
# Helper functions for conversion and selectivity
def conversion(initial: float, remaining:float|list) -> float|list:
    return 1 - remaining / initial

def selectivity(r_desired: float, *r_undesired: float|list) -> float|list:
    sum_of_R_u = np.sum([rate for rate in r_undesired], axis=0) + r_desired
    return r_desired / sum_of_R_u

X_Glu = conversion(C_Glu_feed, Glu_ss)

r1_ss     = k1 * Glu_ss * V_aq                  # mol/s  forward Glucose isomerisation rate
r2_ss     = k2 * Fru_ss * V_aq                  # mol/s  backward Fructose isomerasition rate
r3_ss     = k3 * Fru_ss * V_aq                  # mol/s  forward fructos to HMF production rate
r4_ss     = k4 * HMFaq_ss * V_aq                # mol/s  Humins formation rate
r5_ss     = k5 * HMFaq_ss * V_aq                # mol/s  Levulinic acid and Formic acid production rate
r6_ss     = k6 * eta * HMFp_ss * O2p_ss * V_p   # mol/s  FDCA production rate

S_FDCA_ov  = selectivity(r6_ss, r1_ss, r2_ss, r3_ss, r4_ss, r5_ss)  # selectivity of r6 for overal reaction network
S_FDCA_par = selectivity(r6_ss, r4_ss, r5_ss)   # selectivity of r6 against parallel reaction paths r4 and r5
S_Hum      = selectivity(r4_ss, r5_ss, r6_ss)   # selectivity of r4 against parallel reaction paths r5 and r6
S_LA       = selectivity(r5_ss, r4_ss, r6_ss)   # selectivity of r5 against parallel reaction paths r4 and r6

MW_FDCA, MW_Hum, MW_LA, MW_FA = 168.11, 126.11, 116.12, 46.03   # g/mol

print("="*50)
print("STEADY-STATE RESULTS")
print("="*50)
print(f"Glucose conversion         X  = {X_Glu*100:.1f}%")
print(f"Overal FDCA selectivity    S  = {S_FDCA_ov*100:.1f}%")
print(f"Parallel FDCA selectivity     = {S_FDCA_par*100:.1f}%")
print(f"Humins selectivity            = {S_Hum*100:.2f}%")
print(f"LA+FA selectivity             = {S_LA*100:.2f}%")
print()
print(f"FDCA   = {r6_ss*MW_FDCA/1000*3600:.0f} kg/h")
print(f"Humins = {r4_ss*MW_Hum/1000*3600:.1f} kg/h")
print(f"LA     = {r5_ss*MW_LA/1000*3600:.1f} kg/h")
print(f"FA     = {r5_ss*MW_FA/1000*3600:.1f} kg/h\n")


### 11. CSTR — Dynamic Profiles and Steady-State Performance

Six panels show the transient behaviour from startup to steady state. Vertical dashed lines mark each residence time $\tau$; horizontal dotted lines mark the steady-state value for each variable.

In [ ]:
t_h   = t_plot / 3600
tau_h = tau / 3600

def tau_lines(ax):
    x_max = t_h[-1]
    for i in range(1, int(x_max / tau_h) + 1):
        ax.axvline(i * tau_h, color='grey', lw=0.6, ls='--', alpha=0.5)
    ax.set_xlim(0, x_max)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle(
    'CSTR: Glucose -> Fructose -> HMF -> FDCA\n'
    'T = 140 C,  P = 10 bar,  tau = 1 h,  V = 100 m^3',
    fontsize=12, fontweight='bold')
plt.subplots_adjust(hspace=0.40, wspace=0.35,
                    left=0.07, right=0.97, top=0.88, bottom=0.09)

# Panel 1 -- Sugars (aqueous)
ax = axes[0, 0]
ax.plot(t_h, y_plot[0], label='Glucose')
ax.plot(t_h, y_plot[1], label='Fructose')
ax.axhline(Glu_ss, color='C0', ls=':', lw=1)
ax.axhline(Fru_ss, color='C1', ls=':', lw=1)
ax.set(xlabel='Time (h)', ylabel='Concentration (mol/m^3)', title='Sugars (aq)')
ax.legend(); ax.grid(alpha=0.3); tau_lines(ax)

# Panel 2 -- HMF across phases
ax = axes[0, 1]
ax.plot(t_h, y_plot[2],          label='HMF (aq)')
ax.plot(t_h, y_plot[3],          label='HMF (org)')
ax.plot(t_h, y_plot[4], ls='--', label='HMF (particle)')
ax.axhline(HMFaq_ss,  color='C0', ls=':', lw=1)
ax.axhline(HMForg_ss, color='C1', ls=':', lw=1)
ax.set(xlabel='Time (h)', ylabel='Concentration (mol/m^3)', title='HMF -- All Phases')
ax.legend(); ax.grid(alpha=0.3); tau_lines(ax)

# Panel 3 -- Byproducts
ax = axes[0, 2]
ax.plot(t_h, y_plot[7],          label='Humins (aq)')
ax.plot(t_h, y_plot[8], ls='--', label='LA = FA (aq)')
ax.axhline(Hum_ss, color='C0', ls=':', lw=1)
ax.axhline(LA_ss,  color='C1', ls=':', lw=1)
ax.set(xlabel='Time (h)', ylabel='Concentration (mol/m^3)', title='Byproducts (aq)')
ax.legend(); ax.grid(alpha=0.3); tau_lines(ax)

# Panel 4 -- O2
ax = axes[1, 0]
ax.plot(t_h, y_plot[5],          label='O2 (org)')
ax.plot(t_h, y_plot[6], ls='--', label='O2 (particle)')
ax.axhline(C_O2_sat, color='steelblue', ls=':', lw=1, label=f'C* = {C_O2_sat:.2f}')
ax.set(xlabel='Time (h)', ylabel='Concentration (mol/m^3)', title='O2')
ax.legend(); ax.grid(alpha=0.3); tau_lines(ax)

# Panel 5 -- Conversion and FDCA selectivity
X_t = conversion(C_Glu_feed, y_plot[0]) * 100

r1_t = k1 * y_plot[0] * V_aq
r2_t = k2 * y_plot[1] * V_aq
r3_t = k3 * y_plot[1] * V_aq
r4_t = k4 * y_plot[2] * V_aq
r5_t = k5 * y_plot[2] * V_aq
r6_t = k6 * eta * y_plot[4] * y_plot[6] * V_p

denom_ov  = r1_t + r2_t + r3_t + r4_t + r5_t
denom_par = r4_t + r5_t
with np.errstate(invalid='ignore', divide='ignore'):
    S_t_ov  = np.where(denom_ov  > 0, selectivity(r6_t, r1_t, r2_t, r3_t, r4_t, r5_t) * 100, np.nan)
    S_t_par = np.where(denom_par > 0, selectivity(r6_t, r4_t, r5_t) * 100,                    np.nan)

ax  = axes[1, 1]
ax2 = ax.twinx()
ax3 = ax.twinx()
ax3.spines['right'].set_position(('outward', 60))

l1, = ax.plot(t_h, X_t,      'C0', label='Glu conversion (%)')
l2, = ax2.plot(t_h, S_t_ov,  'C3', ls='--', label='FDCA selectivity overall (%)')
l3, = ax3.plot(t_h, S_t_par, 'C4', ls='--', label='FDCA selectivity parallel (%)')
ax.axhline(X_Glu*100,       color='C0', ls=':', lw=1)
ax2.axhline(S_FDCA_ov*100,  color='C3', ls=':', lw=1)
ax3.axhline(S_FDCA_par*100, color='C4', ls=':', lw=1)

ax2.yaxis.label.set_color('C3')
ax2.tick_params(axis='y', colors='C3')
ax2.spines['right'].set_color('C3')
ax3.yaxis.label.set_color('C4')
ax3.tick_params(axis='y', colors='C4')
ax3.spines['right'].set_color('C4')

ax.set(xlabel='Time (h)', ylabel='Conversion (%)', ylim=(0, 105), title='Performance')
ax2.set_ylabel('Overall selectivity (%)');  ax2.set_ylim(0, 105)
ax3.set_ylabel('Parallel selectivity (%)');
ax.legend([l1, l2, l3], [l.get_label() for l in [l1, l2, l3]], loc='best')
ax.grid(alpha=0.3); tau_lines(ax)

# Panel 6 -- Product rates (log scale)
FDCA_t   = r6_t * 3600 * MW_FDCA/1000
Humins_t = r4_t * 3600 * MW_Hum/1000
LA_t     = r5_t * 3600 * MW_LA/1000
FA_t     = r5_t * 3600 * MW_FA/1000

ax = axes[1, 2]
ax.plot(t_h, FDCA_t,   label=f'FDCA   ({r6_ss*MW_FDCA/1000*3600:.0f} kg/h)')
ax.plot(t_h, Humins_t, label=f'Humins ({r4_ss*MW_Hum/1000*3600:.1f} kg/h)')
ax.plot(t_h, LA_t,     ls='--', label=f'LA ({r5_ss*MW_LA/1000*3600:.1f} kg/h)')
ax.plot(t_h, FA_t,     ls='-.', label=f'FA ({r5_ss*MW_FA/1000*3600:.1f} kg/h)')
ax.set_yscale('log')
ax.set(xlabel='Time (h)', ylabel='Production rate (kg/h)', title='Product Rates')
ax.legend(fontsize=8.5); ax.grid(alpha=0.3); tau_lines(ax)

plt.tight_layout()
plt.show()

### 12. CSTR — Parameter Study
Here we look at how small changes parameters such as $T, P_{\text{O}_2}, \varepsilon_\text{org}$ lead to changes in the conversion and if they are impactfull in a significant way. 

1) make a list of parameters we want to test and change
2) make a range and number of points on said range to change up the parameters
3) make the odes have some parameters that are fixed through the scope insertion from outside function to inside at function initialisation.
4) other parameters are set via arguments of the function -> making them changable and can be defined to have a standard value at function creation filled in, eg. f(t,y, params, x=X)
5) plot the results and have the conclusions

---
## Part B — SDR (Spinning Disc Reactor)

The SDR is approximated as **N ideal CSTRs in series**, giving a narrower RTD / plug-flow-like behaviour.

Key differences from the CSTR:
- **Multistage hydrodynamics**: stagewise solution marching from inlet to outlet.
- **SDR mass transfer coefficients**: taken from the characteristic SDR ranges in the slides
  (gas–liquid ≈ 10 s⁻¹, liquid–liquid ≈ 300 s⁻¹, liquid–solid ≈ 150–300 s⁻¹),
  clearly higher than the CSTR correlations above.
- **No impeller / agitator**: mass transfer driven by rotor geometry, not modelled from first principles here.

All kinetics, fluid properties, and operating conditions are shared with Part A.


### 13. SDR Design Basis — N Stages in Series

The same total reactive volume (100 m³) and residence time (1 h) as the CSTR are used so that
the reactor concept is the only major difference.

The SDR is split into **N stages in series**; each stage has volume = total / N.
As N → ∞ the RTD approaches plug flow.


In [ ]:
# SDR hydrodynamic representation — splits the shared reactor volume into N stages
N_stages = 20                            # plug-flow approximation
tau_total = tau                          # alias: shared residence time (s)
tau_stage = tau / N_stages               # s  per-stage residence time

V_aq_stage  = V_aq  / N_stages          # m_aq^3  per stage
V_org_stage = V_org / N_stages          # m_org^3 per stage
V_p_stage   = V_p   / N_stages          # m_s^3   per stage

print("SDR in-series approximation")
print("-" * 40)
print(f"N_stages    = {N_stages}")
print(f"tau_stage   = {tau_stage:.1f} s ({tau_stage/60:.2f} min)")
print(f"V_aq_stage  = {V_aq_stage:.3f} m^3")
print(f"V_org_stage = {V_org_stage:.3f} m^3")
print(f"V_p_stage   = {V_p_stage:.3f} m^3")


### 14. SDR Mass Transfer Coefficients

For a rotor–stator SDR the mass transfer is characterised by the slide-based ranges:
- gas–liquid: order of **10 s⁻¹**
- liquid–liquid: order of **300 s⁻¹**
- liquid–solid: **1–300 s⁻¹**

Representative base-case values are used below. These are stored under `*_sdr` names
to avoid overwriting the CSTR correlation-based values above.


In [ ]:
# Representative rotor-stator SDR volumetric mass-transfer coefficients
# Clearly higher than the CSTR values; within the slide-based SDR range.
kLa_GL_sdr     = 10.0    # s^-1  O2:  gas  -> organic
kLa_LL_sdr     = 300.0   # s^-1  HMF: aq   -> organic
kLa_LS_HMF_sdr = 150.0   # s^-1  HMF: org  -> catalyst
kLa_LS_O2_sdr  = 300.0   # s^-1  O2:  org  -> catalyst

print("SDR base-case mass transfer coefficients")
print("-" * 48)
print(f"kLa_GL_sdr     = {kLa_GL_sdr:.2f} s^-1   (CSTR: {kLa_GL:.4f} s^-1)")
print(f"kLa_LL_sdr     = {kLa_LL_sdr:.2f} s^-1  (CSTR: {kLa_LL_HMF:.4f} s^-1)")
print(f"kLa_LS_HMF_sdr = {kLa_LS_HMF_sdr:.2f} s^-1  (CSTR: {kLa_LS_HMF:.2f} s^-1)")
print(f"kLa_LS_O2_sdr  = {kLa_LS_O2_sdr:.2f} s^-1  (CSTR: {kLa_LS_O2:.2f} s^-1)")


### 15. SDR Stage Model

Each stage is a small CSTR. Inlet concentrations come from the previous stage outlet.
The full SDR profile is obtained by marching stage-by-stage from inlet to outlet.

> **Note:** The SDR stage ODE uses 9 state variables (no explicit FDCA tracking in each stage).
> FDCA production is accumulated from the per-stage reaction rate $r_6 \cdot V_{p,\text{stage}}$.


In [ ]:
def sdr_stage_odes(t, y, aq_in, org_in, pars):
    (
        k1, k2, k3, k4, k5, k6, eta,
        kLa_GL, kLa_LL, kLa_LS_HMF, kLa_LS_O2,
        m_AO, C_O2_sat,
        F_aq, F_org,
        V_aq_stage, V_org_stage, V_p_stage
    ) = pars

    Glu, Fru, HMF_aq, HMF_org, HMF_p, O2_org, O2_p, Hum, LA = [max(v, 0.0) for v in y]

    # Reaction rates
    r1 = k1 * Glu
    r2 = k2 * Fru
    r3 = k3 * Fru
    r4 = k4 * HMF_aq
    r5 = k5 * HMF_aq
    r6 = k6 * eta * HMF_p * O2_p

    # Mass transfer fluxes
    J_GL  = kLa_GL * (C_O2_sat - O2_org)
    J_LL  = kLa_LL * (HMF_aq - m_AO * HMF_org)
    J_HMF = kLa_LS_HMF * (HMF_org - HMF_p)
    J_O2  = kLa_LS_O2  * (O2_org - O2_p)

    # Mole balances
    dGlu    = (F_aq / V_aq_stage)  * (aq_in[0] - Glu)    + (-r1 + r2)
    dFru    = (F_aq / V_aq_stage)  * (aq_in[1] - Fru)    + (r1 - r2 - r3)
    dHMF_aq = (F_aq / V_aq_stage)  * (aq_in[2] - HMF_aq) + (r3 - r4 - r5) - J_LL

    dHMF_org = (F_org / V_org_stage) * (org_in[0] - HMF_org) + J_LL * (V_aq_stage / V_org_stage) - J_HMF
    dO2_org  = (F_org / V_org_stage) * (org_in[1] - O2_org)  + J_GL - J_O2

    dHMF_p = J_HMF * (V_org_stage / V_p_stage) - r6
    dO2_p  = J_O2  * (V_org_stage / V_p_stage) - r6

    dHum = (F_aq / V_aq_stage) * (aq_in[3] - Hum) + r4
    dLA  = (F_aq / V_aq_stage) * (aq_in[4] - LA)  + r5

    return [dGlu, dFru, dHMF_aq, dHMF_org, dHMF_p, dO2_org, dO2_p, dHum, dLA]


def run_sdr_model(
    N_stages=20,
    tau_total=3600.0,
    kLa_GL=10.0,
    kLa_LL=300.0,
    kLa_LS_HMF=150.0,
    kLa_LS_O2=300.0,
    eps_g=0.20,
    eps_s=0.10,
    eps_aq=1/3,
):
    eps_l = 1.0 - eps_g - eps_s
    eps_org = 1.0 - eps_aq

    V_g   = eps_g * V_total
    V_p   = eps_s * V_total
    V_liq = eps_l * V_total
    V_aq  = eps_aq * V_liq
    V_org = eps_org * V_liq

    F_aq  = V_aq / tau_total
    F_org = V_org / tau_total

    V_aq_stage  = V_aq  / N_stages
    V_org_stage = V_org / N_stages
    V_p_stage   = V_p   / N_stages
    tau_stage   = tau_total / N_stages

    pars = (
        k1, k2, k3, k4, k5, k6, eta,
        kLa_GL, kLa_LL, kLa_LS_HMF, kLa_LS_O2,
        m_AO, C_O2_sat,
        F_aq, F_org,
        V_aq_stage, V_org_stage, V_p_stage
    )

    aq_in = np.array([C_Glu_feed, 0.0, 0.0, 0.0, 0.0], dtype=float)
    org_in = np.array([0.0, C_O2_sat], dtype=float)

    stage_ss = []
    stage_dyn = []
    r6_stage = []

    for _ in range(N_stages):
        y0 = np.array([
            aq_in[0],       # Glucose (aq) in CSTR i
            aq_in[1],       # Fructose (aq) in CSTR i 
            aq_in[2],       # HMF (aq)   in CSTR i 
            org_in[0],      # HMF (org)  in CSTR i 
            0.0,            # HMF (cat)  in CSTR i
            org_in[1],      # O2  (org)  in CSTR i
            org_in[1],      # O2  (cat)  in CSTR i
            aq_in[3],       # Humine (aq)  in CSTR i
            aq_in[4]        # LA & Fa (aq)  in CSTR i
        ], dtype=float)

        sol = solve_ivp(
            lambda t, y: sdr_stage_odes(t, y, aq_in, org_in, pars),
            [0.0, sim_time * tau],
            y0,
            method="BDF",
            rtol=1e-8,
            atol=1e-10,
            dense_output=True
        )

        ss = sol.y[:, -1]

        stage_ss.append(ss)
        stage_dyn.append(sol)

        rate_r6 = k6 * eta * max(ss[4], 0.0) * max(ss[6], 0.0)
        r6_stage.append(rate_r6)

        aq_in = np.array([ss[0], ss[1], ss[2], ss[7], ss[8]])
        org_in = np.array([ss[3], ss[5]])

    stage_ss = np.array(stage_ss)
    r6_stage = np.array(r6_stage)

    ss_out = stage_ss[-1, :]

    X_Glu = conversion(C_Glu_feed, ss_out[0])
    
    F_FDCA_total = np.sum(r6_stage * V_p_stage)
    F_Hum_out = F_aq * ss_out[7]
    F_LA_out  = F_aq * ss_out[8]
    F_FA_out  = F_LA_out
    F_Glu_rxd = F_aq * C_Glu_feed * X_Glu

    if F_Glu_rxd > 1e-15:
        S_FDCA = F_FDCA_total / F_Glu_rxd
        S_Hum  = F_Hum_out / F_Glu_rxd
        S_LA   = F_LA_out  / F_Glu_rxd
    else:
        S_FDCA = 0.0
        S_Hum  = 0.0
        S_LA   = 0.0

    results = {
        "N_stages": N_stages,
        "tau_total": tau_total,
        "tau_stage": tau_stage,
        "F_aq": F_aq,
        "F_org": F_org,
        "V_aq": V_aq,
        "V_org": V_org,
        "V_p": V_p,
        "V_aq_stage": V_aq_stage,
        "V_org_stage": V_org_stage,
        "V_p_stage": V_p_stage,
        "stage_ss": stage_ss,
        "stage_dyn": stage_dyn,
        "r6_stage": r6_stage,
        "outlet": ss_out,
        "F_FDCA_total": F_FDCA_total,
        "F_Hum_out": F_Hum_out,
        "F_LA_out": F_LA_out,
        "F_FA_out": F_FA_out,
        "F_Glu_rxd": F_Glu_rxd,
        "X_Glu": X_Glu,
        "S_FDCA": S_FDCA,
        "S_Hum": S_Hum,
        "S_LA": S_LA,
        "kLa_GL": kLa_GL,
        "kLa_LL": kLa_LL,
        "kLa_LS_HMF": kLa_LS_HMF,
        "kLa_LS_O2": kLa_LS_O2,
        "eps_g": eps_g,
        "eps_s": eps_s,
        "eps_aq": eps_aq,
    }
    return results


### 16. SDR Base-case Run

In [ ]:
res = run_sdr_model(
    N_stages=N_stages,
    tau_total=tau,
    kLa_GL=kLa_GL_sdr,
    kLa_LL=kLa_LL_sdr,
    kLa_LS_HMF=kLa_LS_HMF_sdr,
    kLa_LS_O2=kLa_LS_O2_sdr,
    eps_g=eps_g,
    eps_s=eps_s,
    eps_aq=eps_aq,
)

stage_ss = res["stage_ss"]
ss = res["outlet"]

Glu_ss_sdr, Fru_ss_sdr, HMFaq_ss_sdr, HMForg_ss_sdr, HMFp_ss_sdr,     O2org_ss_sdr, O2p_ss_sdr, Hum_ss_sdr, LA_ss_sdr = ss

print("Base-case SDR outlet concentrations")
print("-" * 48)
print(f"Glu     = {Glu_ss_sdr:.3f} mol/m^3")
print(f"Fru     = {Fru_ss_sdr:.3f} mol/m^3")
print(f"HMF_aq  = {HMFaq_ss_sdr:.5f} mol/m^3")
print(f"HMF_org = {HMForg_ss_sdr:.5f} mol/m^3")
print(f"HMF_p   = {HMFp_ss_sdr:.5f} mol/m^3")
print(f"O2_org  = {O2org_ss_sdr:.5f} mol/m^3")
print(f"O2_p    = {O2p_ss_sdr:.5f} mol/m^3")
print(f"Humins  = {Hum_ss_sdr:.5f} mol/m^3")
print(f"LA = FA = {LA_ss_sdr:.5f} mol/m^3")


### 17. SDR Performance Metrics

In [ ]:
MW_FDCA = 168.11
MW_Hum = 126.11
MW_LA = 116.12
MW_FA = 46.03

FDCA_kgh = res["F_FDCA_total"] * MW_FDCA / 1000.0 * 3600.0
Hum_kgh  = res["F_Hum_out"]    * MW_Hum  / 1000.0 * 3600.0
LA_kgh   = res["F_LA_out"]     * MW_LA   / 1000.0 * 3600.0
FA_kgh   = res["F_FA_out"]     * MW_FA   / 1000.0 * 3600.0

print("=" * 56)
print("SDR BASE-CASE RESULTS")
print("=" * 56)
print(f"Number of stages            = {res['N_stages']}")
print(f"Total residence time        = {res['tau_total']/3600:.2f} h")
print(f"Residence time per stage    = {res['tau_stage']:.1f} s")
print()
print(f"Glucose conversion          = {res['X_Glu']*100:.2f} %")
print(f"FDCA selectivity            = {res['S_FDCA']*100:.2f} %")
print(f"Humins selectivity          = {res['S_Hum']*100:.3f} %")
print(f"LA+FA selectivity           = {res['S_LA']*100:.3f} %")
print()
print(f"FDCA production             = {FDCA_kgh:.1f} kg/h")
print(f"Humins production           = {Hum_kgh:.3f} kg/h")
print(f"LA production               = {LA_kgh:.3f} kg/h")
print(f"FA production               = {FA_kgh:.3f} kg/h")


### 18. SDR Stagewise Profiles

The plots below show how concentrations evolve from stage 1 to stage N —
the SDR analogue of moving along the flow path in a plug-flow reactor.


In [ ]:
stage_index = np.arange(1, res["N_stages"] + 1)
fdca_stage_kgh = res["r6_stage"] * res["V_p_stage"] * MW_FDCA / 1000.0 * 3600.0
fdca_cum_kgh = np.cumsum(fdca_stage_kgh)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle(
    "SDR (N CSTRs in series): stagewise concentration and performance profiles",
    fontsize=12, fontweight="bold"
)
plt.subplots_adjust(hspace=0.38, wspace=0.30, top=0.90)

# Sugars
ax = axes[0, 0]
ax.plot(stage_index, stage_ss[:, 0], label="Glucose")
ax.plot(stage_index, stage_ss[:, 1], label="Fructose")
ax.set_xlabel("Stage number")
ax.set_ylabel("Concentration (mol/m³)")
ax.set_title("Sugars (aqueous)")
ax.grid(alpha=0.3)
ax.legend()

# HMF
ax = axes[0, 1]
ax.plot(stage_index, stage_ss[:, 2], label="HMF (aq)")
ax.plot(stage_index, stage_ss[:, 3], label="HMF (org)")
ax.plot(stage_index, stage_ss[:, 4], "--", label="HMF (particle)")
ax.set_xlabel("Stage number")
ax.set_ylabel("Concentration (mol/m³)")
ax.set_title("HMF across phases")
ax.grid(alpha=0.3)
ax.legend()

# O2
ax = axes[0, 2]
ax.plot(stage_index, stage_ss[:, 5], label="O₂ (org)")
ax.plot(stage_index, stage_ss[:, 6], "--", label="O₂ (particle)")
ax.axhline(C_O2_sat, color="steelblue", ls=":", lw=1, label=f"C* = {C_O2_sat:.2f}")
ax.set_xlabel("Stage number")
ax.set_ylabel("Concentration (mol/m³)")
ax.set_title("O₂ profiles")
ax.grid(alpha=0.3)
ax.legend(loc='lower right')

# Byproducts
ax = axes[1, 0]
ax.plot(stage_index, stage_ss[:, 7], label="Humins")
ax.plot(stage_index, stage_ss[:, 8], "--", label="LA = FA")
ax.set_xlabel("Stage number")
ax.set_ylabel("Concentration (mol/m³)")
ax.set_title("Byproducts (aqueous)")
ax.grid(alpha=0.3)
ax.legend()

# Conversion / selectivity
V_aq_stage = res["V_aq_stage"]
V_p_stage  = res["V_p_stage"]

r1_stage = k1 * stage_ss[:, 0] * V_aq_stage
r2_stage = k2 * stage_ss[:, 1] * V_aq_stage
r3_stage = k3 * stage_ss[:, 1] * V_aq_stage
r4_stage = k4 * stage_ss[:, 2] * V_aq_stage
r5_stage = k5 * stage_ss[:, 2] * V_aq_stage
r6_stage = res["r6_stage"] * V_p_stage

cum_r6 = np.cumsum(r6_stage)
cum_r1 = np.cumsum(r1_stage)
cum_r2 = np.cumsum(r2_stage)
cum_r3 = np.cumsum(r3_stage)
cum_r4 = np.cumsum(r4_stage)
cum_r5 = np.cumsum(r5_stage)

X_stage     = conversion(C_Glu_feed, stage_ss[:, 0]) * 100
S_ov_stage  = selectivity(cum_r6, cum_r1, cum_r2, cum_r3, cum_r4, cum_r5) * 100
S_par_stage = selectivity(cum_r6, cum_r4, cum_r5) * 100

ax  = axes[1, 1]
ax2 = ax.twinx()

l1, = ax.plot(stage_index, X_stage,     "C0",   label="Glu conversion")
l2, = ax2.plot(stage_index, S_ov_stage,  "C3--", label="FDCA selectivity overall")
l3, = ax2.plot(stage_index, S_par_stage, "C4:",  label="FDCA selectivity parallel")
ax.axhline(X_stage[-1],      color="C0", ls=":", lw=1)
ax2.axhline(S_ov_stage[-1],  color="C3", ls=":", lw=1)
ax2.axhline(S_par_stage[-1], color="C4", ls=":", lw=1)

ax2.yaxis.label.set_color("C3")
ax2.tick_params(axis="y", colors="C3")
ax2.spines["right"].set_color("C3")

ax.set_xlabel("Stage number")
ax.set_ylabel("Conversion (%)")
ax2.set_ylabel("Selectivity (%)")
ax.set_ylim(0, 105); ax2.set_ylim(0, 105)
ax.set_title("Performance")
ax.grid(alpha=0.3)
ax.legend([l1, l2, l3], [l.get_label() for l in [l1, l2, l3]], loc="right")

# FDCA generation
ax = axes[1, 2]
ax.plot(stage_index, fdca_stage_kgh, label="FDCA per stage")
ax.plot(stage_index, fdca_cum_kgh, "--", label="Cumulative FDCA")
ax.set_xlabel("Stage number")
ax.set_ylabel("kg/h")
ax.set_title("FDCA production")
ax.grid(alpha=0.3)
ax.legend()

plt.show()

In [ ]:
# Transient behaviour of the last SDR stage
# The last stage receives a fixed inlet (stage N-1 SS) and integrates to its own SS,
# making it the cleanest representation of "true" CSTR-like transient dynamics in the SDR.

last_dyn   = res["stage_dyn"][-1]
tau_s      = res["tau_stage"]           # residence time of one stage [s]
t_last     = np.linspace(0, last_dyn.t[-1], 2000)
y_last     = last_dyn.sol(t_last)
t_h        = t_last / 3600              # real time in hours
ss_last    = res["stage_ss"][-1]        # steady-state of last stage

# Instantaneous rates of the last stage along its transient [mol/s]
r1_l = k1 * y_last[0] * res["V_aq_stage"]
r2_l = k2 * y_last[1] * res["V_aq_stage"]
r3_l = k3 * y_last[1] * res["V_aq_stage"]
r4_l = k4 * y_last[2] * res["V_aq_stage"]
r5_l = k5 * y_last[2] * res["V_aq_stage"]
r6_l = k6 * eta * y_last[4] * y_last[6] * res["V_p_stage"]

X_last = conversion(C_Glu_feed, y_last[0]) * 100

denom_ov  = r1_l + r2_l + r3_l + r4_l + r5_l
denom_par = r4_l + r5_l
with np.errstate(invalid='ignore', divide='ignore'):
    S_ov_last  = np.where(denom_ov  > 0, selectivity(r6_l, r1_l, r2_l, r3_l, r4_l, r5_l) * 100, np.nan)
    S_par_last = np.where(denom_par > 0, selectivity(r6_l, r4_l, r5_l) * 100,                    np.nan)

# Whole-SDR steady-state rates — summed over all N stages [mol/s]
V_aq_s = res["V_aq_stage"]
V_p_s  = res["V_p_stage"]
R1_ss = np.sum(k1 * stage_ss[:, 0] * V_aq_s)
R2_ss = np.sum(k2 * stage_ss[:, 1] * V_aq_s)
R3_ss = np.sum(k3 * stage_ss[:, 1] * V_aq_s)
R4_ss = np.sum(k4 * stage_ss[:, 2] * V_aq_s)
R5_ss = np.sum(k5 * stage_ss[:, 2] * V_aq_s)
R6_ss = np.sum(k6 * eta * stage_ss[:, 4] * stage_ss[:, 6] * V_p_s)

X_sdr_ss     = res["X_Glu"] * 100
S_ov_sdr_ss  = selectivity(R6_ss, R1_ss, R2_ss, R3_ss, R4_ss, R5_ss) * 100
S_par_sdr_ss = selectivity(R6_ss, R4_ss, R5_ss) * 100

def ss_line(ax, val, color):
    ax.axhline(val, color=color, ls=':', lw=1)

# Match x-axis to CSTR view window: 1 full reactor residence time = tau hours
tau_s_h = tau_s / 3600   # τ_stage in hours
x_max_h = tau   / 3600   # = 1 h, same as CSTR transient plots

def tau_vlines(ax):
    i = 1
    while i * tau_s_h <= x_max_h:
        ax.axvline(i * tau_s_h, color='grey', lw=0.6, ls='--', alpha=0.5)
        i += 1
    ax.set_xlim(0, x_max_h)
    ax.xaxis.set_major_locator(plt.MultipleLocator(0.2))
    ax.grid(alpha=0.3)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle(
    f'SDR — Transient\n$\\tau_{{stage}}$ = {tau_s:.0f} s, $\\tau$ = {tau:.0f} s',
    fontsize=12, fontweight='bold')
plt.subplots_adjust(hspace=0.40, wspace=0.35, left=0.07, right=0.97, top=0.88, bottom=0.09)

# Panel 1 -- Sugars
ax = axes[0, 0]
ax.plot(t_h, y_last[0], label='Glucose')
ax.plot(t_h, y_last[1], label='Fructose')
ss_line(ax, ss_last[0], 'C0'); ss_line(ax, ss_last[1], 'C1')
ax.set(xlabel='Time (h)', ylabel='Concentration (mol/m³)', title='Sugars (aq)')
ax.legend(); tau_vlines(ax)

# Panel 2 -- HMF across phases
ax = axes[0, 1]
ax.plot(t_h, y_last[2],          label='HMF (aq)')
ax.plot(t_h, y_last[3],          label='HMF (org)')
ax.plot(t_h, y_last[4], ls='--', label='HMF (particle)')
ss_line(ax, ss_last[2], 'C0'); ss_line(ax, ss_last[3], 'C1')
ax.set(xlabel='Time (h)', ylabel='Concentration (mol/m³)', title='HMF — All Phases')
ax.legend(); tau_vlines(ax)

# Panel 3 -- Byproducts
ax = axes[0, 2]
ax.plot(t_h, y_last[7],          label='Humins (aq)')
ax.plot(t_h, y_last[8], ls='--', label='LA = FA (aq)')
ss_line(ax, ss_last[7], 'C0'); ss_line(ax, ss_last[8], 'C1')
ax.set(xlabel='Time (h)', ylabel='Concentration (mol/m³)', title='Byproducts (aq)')
ax.legend(); tau_vlines(ax)

# Panel 4 -- O2
ax = axes[1, 0]
ax.plot(t_h, y_last[5],          label='O₂ (org)')
ax.plot(t_h, y_last[6], ls='--', label='O₂ (particle)')
ax.axhline(C_O2_sat, color='steelblue', ls=':', lw=1, label=f'C* = {C_O2_sat:.2f}')
ax.set(xlabel='Time (h)', ylabel='Concentration (mol/m³)', title='O₂')
ax.legend(); tau_vlines(ax)

# Panel 5 -- Conversion and selectivity
ax  = axes[1, 1]
ax2 = ax.twinx()

l1, = ax.plot(t_h,  X_last,     'C0',   label='Glu conversion (%)')
l2, = ax2.plot(t_h, S_ov_last,  'C3--', label='FDCA selectivity overall (%)')
l3, = ax2.plot(t_h, S_par_last, 'C4:',  label='FDCA selectivity parallel (%)')
ss_line(ax,  X_sdr_ss,     'C0')
ss_line(ax2, S_ov_sdr_ss,  'C3')
ss_line(ax2, S_par_sdr_ss, 'C4')

ax2.yaxis.label.set_color('C3')
ax2.tick_params(axis='y', colors='C3')
ax2.spines['right'].set_color('C3')

ax.set(xlabel='Time (h)', ylabel='Conversion (%)', ylim=(0, 105), title='Performance')
ax2.set_ylabel('Selectivity (%)'); ax2.set_ylim(0, 105)
ax.legend([l1, l2, l3], [l.get_label() for l in [l1, l2, l3]], loc='best')
tau_vlines(ax)

# Panel 6 -- Product rates (log scale), matching CSTR panel style
ax = axes[1, 2]
ax.plot(t_h, r6_l * 3600 * MW_FDCA / 1000, label=f'FDCA   ({R6_ss*MW_FDCA/1000*3600:.0f} kg/h)')
ax.plot(t_h, r4_l * 3600 * MW_Hum  / 1000, label=f'Humins ({R4_ss*MW_Hum /1000*3600:.1f} kg/h)')
ax.plot(t_h, r5_l * 3600 * MW_LA   / 1000, ls='--', label=f'LA ({R5_ss*MW_LA/1000*3600:.1f} kg/h)')
ax.plot(t_h, r5_l * 3600 * MW_FA   / 1000, ls='-.', label=f'FA ({R5_ss*MW_FA/1000*3600:.1f} kg/h)')
ax.set_yscale('log')
ax.set(xlabel='Time (h)', ylabel='Production rate (kg/h)', title='Product Rates')
ax.legend(fontsize=8.5); tau_vlines(ax)

plt.tight_layout()
plt.show()


### 19. SDR Rate-Determining Step (RDS) Analysis

Same maximum-driving-force approach as the CSTR RDS section, but using the SDR kLa values
(retrieved from the results dict so any `run_sdr_model` run can be re-analysed).


In [ ]:
K_eq = k1 / k2
C_Fru_max = K_eq / (1.0 + K_eq) * C_Glu_feed
C_HMF_aq_max = C_Glu_feed
C_HMF_org_max = m_AO * C_HMF_aq_max

rds = {
    "k1  Glu->Fru  (aq)": k1 * C_Glu_feed * res["V_aq"],
    "k3  Fru->HMF  (aq)": k3 * C_Fru_max * res["V_aq"],
    "LL  HMF aq->org":    res["kLa_LL"] * C_HMF_aq_max * res["V_aq"],
    "LS  HMF org->cat":   res["kLa_LS_HMF"] * C_HMF_org_max * res["V_org"],
    "GL  O2  gas->org":   res["kLa_GL"] * C_O2_sat * res["V_org"],
    "LS  O2  org->cat":   res["kLa_LS_O2"] * C_O2_sat * res["V_org"],
    "k6*eta reaction":    k6 * eta * C_HMF_org_max * C_O2_sat * res["V_p"],
}

min_rate = min(rds.values())

print("=" * 58)
print("SDR RDS -- MAXIMUM DRIVING FORCE [mol/s]")
print("=" * 58)
for name, rate in rds.items():
    tag = "  <-- BOTTLENECK" if rate == min_rate else ""
    print(f"{name:24s} {rate:12.3e}{tag}")


### 20. SDR N-stages Sensitivity

As N increases the RTD narrows and the SDR approaches plug-flow behaviour.


In [ ]:
rows = []
for N in [1, 5, 10, 20, 40]:
    rr = run_sdr_model(
        N_stages=N,
        tau_total=tau,
        kLa_GL=kLa_GL_sdr,
        kLa_LL=kLa_LL_sdr,
        kLa_LS_HMF=kLa_LS_HMF_sdr,
        kLa_LS_O2=kLa_LS_O2_sdr,
        eps_g=eps_g,
        eps_s=eps_s,
        eps_aq=eps_aq,
    )
    rows.append({
        "N_stages": N,
        "tau_stage_s": rr["tau_stage"],
        "X_Glu_%": rr["X_Glu"] * 100.0,
        "S_FDCA_%": rr["S_FDCA"] * 100.0,
        "FDCA_kg_h": rr["F_FDCA_total"] * MW_FDCA / 1000.0 * 3600.0,
        "Humins_kg_h": rr["F_Hum_out"] * MW_Hum / 1000.0 * 3600.0,
    })

sens_df = pd.DataFrame(rows)
sens_df


### 21. SDR Summary

In [ ]:
summary = {
    "X_Glu_percent": round(res["X_Glu"] * 100, 2),
    "S_FDCA_percent": round(res["S_FDCA"] * 100, 2),
    "FDCA_kg_per_h": round(FDCA_kgh, 2),
    "Humins_kg_per_h": round(Hum_kgh, 4),
    "LA_kg_per_h": round(LA_kgh, 4),
    "FA_kg_per_h": round(FA_kgh, 4),
    "RDS": min(rds, key=rds.get),
}
summary


---
## Part C — Comparison

### Residence Time Distribution (RTD)
$$E(t) = \frac{C_{\text{out}}(t)}{\int_0^\infty C_{\text{out}}(t) dt}$$

In [ ]:
def RTD(concentration:list, delta_t:float) -> list:
    integral = np.sum(concentration) * delta_t
    return concentration / integral